# 06 - Final holdout evaluation and return scenarios

This notebook evaluates the frozen specification from notebooks 01-05: logistic regression with `C = 1`, and a standing weekly queue ranked by cancellation risk times booked value. The urgency-boosted variant is carried along as a reported sensitivity, not as the selected policy.

No probability calibration is added: `p_cancel` is used as a booking-time risk score, and the combined score is a priority index rather than an expected-dollar estimate.

Cancellation loss is valued under the two bounds fixed in notebook 02 - linear resale recovery within 30 days, and no resale at all - because how much of a canceled room is resold is a business fact this dataset does not contain, and the program's business case turns out to depend on it more than on the model.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "hotel_bookings.csv").exists()
)
sys.path.insert(0, str(PROJECT_ROOT / "modeling"))

import evaluation_policy as evaluation
from models import make_logistic_pipeline
from prepare_data import (
    RAW_DATA_PATH, complete_180_rows, model_features, prepare_data,
)

pd.set_option("display.max_columns", 60)

## 1. Reproduce the frozen model

The model is fitted on the same 43,264 resolved training-period bookings used for validation model selection. Validation chose the model and policy; its labels are not used to change the final fit. This keeps the final holdout comparison directly tied to the experiment defined in notebook 02.

Only the approved booking-time features enter the model. Outcome and post-booking fields remain excluded.

In [2]:
raw = pd.read_csv(RAW_DATA_PATH)
prepared = prepare_data(raw)
modeling_population = complete_180_rows(prepared)

validation_population = modeling_population.loc[
    modeling_population["booking_period_split"].eq("validation")
].copy()
holdout_population = modeling_population.loc[
    modeling_population["booking_period_split"].eq("holdout")
].copy()
validation_start = validation_population["booking_week_start"].min()

nominal_train = modeling_population.loc[
    modeling_population["booking_period_split"].eq("train")
].copy()
model_fit_population = nominal_train.loc[
    nominal_train["arrival_date"].lt(validation_start)
].copy()

selected_model = make_logistic_pipeline(c_value=1.0)
selected_model.fit(
    model_features(model_fit_population),
    model_fit_population["is_canceled"],
)

holdout_scored = holdout_population.copy()
holdout_scored["p_cancel"] = selected_model.predict_proba(
    model_features(holdout_population)
)[:, 1]

display(
    pd.Series(
        {
            "model fitting rows": len(model_fit_population),
            "holdout rows": len(holdout_population),
            "holdout booking weeks": holdout_population[
                "booking_week_start"
            ].nunique(),
            "first holdout booking week": holdout_population[
                "booking_week_start"
            ].min(),
            "last holdout booking week": holdout_population[
                "booking_week_start"
            ].max(),
        },
        name="final evaluation",
    ).to_frame()
)

,final evaluation
model fitting rows,43264
holdout rows,17075
holdout booking weeks,13
first holdout booking week,2016-11-28 00:00:00
last holdout booking week,2017-02-20 00:00:00


## 2. Run the weekly policy

The business policy is evaluated exactly as defined earlier: a fresh standing queue, one list each Monday, at most 150 calls per week, one call per booking, and uncalled eligible bookings carried forward. A booking's model score stays fixed from booking time.

Three rankings are reported. Risk-only is the model-quality reference, risk x value is the selected policy, and risk x value x urgency is the sensitivity arm that validation rejected. The repeated random queue is the capacity-matched baseline.

In the result tables, a cancellation outcome means `is_canceled = 1`, which includes both `Canceled` and `No-Show`; the final recommendation reports them separately. Realized loss uses the actual cancellation date and the 30-day resale assumption; full-value loss assumes no resale. Neither valuation is visible to the ranking.

In [3]:
SELECTED_POLICY = "risk_and_value"
REFERENCE_POLICY = "risk_only"

policy_labels = {
    "risk_only": "risk only (model reference)",
    "risk_and_value": "risk x booking value (selected)",
    "risk_value_and_urgency": "risk x value x urgency (sensitivity)",
}
holdout_policy_results = {}
summary_rows = []

for policy, label in policy_labels.items():
    calls, weekly, aggregate, final_queue = (
        evaluation.simulate_weekly_policy(
            holdout_scored,
            "holdout",
            policy=policy,
            urgency_window_days=30,
        )
    )
    holdout_policy_results[policy] = {
        "calls": calls,
        "weekly": weekly,
        "aggregate": aggregate,
        "final_queue": final_queue,
    }
    summary_rows.append(
        {
            "policy": label,
            "calls": len(calls),
            "cancellations reached": int(
                aggregate["cancellations reached"]
            ),
            "mean weekly precision@150": aggregate[
                "mean weekly queue precision at capacity"
            ],
            "realized loss reached": aggregate["realized loss reached"],
            "full value loss reached": aggregate[
                "full value loss reached"
            ],
            "average days to arrival": calls[
                "days_to_arrival_at_call"
            ].mean(),
            "capacity utilization": aggregate[
                "mean capacity utilization"
            ],
            "bookings waiting after final decision": aggregate[
                "bookings still waiting after final decision"
            ],
        }
    )

holdout_summary = pd.DataFrame(summary_rows).set_index("policy")
display(
    holdout_summary.style.format(
        {
            "mean weekly precision@150": "{:.2%}",
            "realized loss reached": "{:,.0f}",
            "full value loss reached": "{:,.0f}",
            "average days to arrival": "{:.1f}",
            "capacity utilization": "{:.2%}",
            "bookings waiting after final decision": "{:,.0f}",
        }
    )
)

model_metrics = evaluation.model_metric_summary(
    holdout_scored["is_canceled"], holdout_scored["p_cancel"]
)
display(model_metrics.map(lambda value: f"{value:.4f}").to_frame("holdout"))

random_results = evaluation.repeated_random_baseline(
    holdout_scored, "holdout", repeats=100
)
random_metric = "mean weekly queue precision at capacity"
display(
    pd.Series(
        {
            "random mean": random_results[random_metric].mean(),
            "random 5th percentile": random_results[
                random_metric
            ].quantile(0.05),
            "random 95th percentile": random_results[
                random_metric
            ].quantile(0.95),
        },
        name="mean weekly precision@150",
    ).map(lambda value: f"{value:.2%}").to_frame()
)

display(evaluation.pre_call_attrition(holdout_scored).to_frame())

,calls,cancellations reached,mean weekly precision@150,realized loss reached,full value loss reached,average days to arrival,capacity utilization,bookings waiting after final decision
policy,,,,,,,,
risk only (model reference),1950,1278,65.54%,"106,273","693,506",107.0,100.00%,"5,721"
risk x booking value (selected),1950,1146,58.77%,"130,353","803,697",112.0,100.00%,"5,621"
risk x value x urgency (sensitivity),1950,1119,57.38%,"134,221","792,516",106.8,100.00%,"5,672"


,holdout
average_precision,0.8414
roc_auc,0.8829
brier_score,0.1318


,mean weekly precision@150
random mean,25.99%
random 5th percentile,24.15%
random 95th percentile,27.49%


,bookings
canceled before first Monday,3207
arrived by first Monday,2286
available at first opportunity,11582


### Weekly behavior of the selected policy

This table makes capacity use and week-to-week variation visible. The final queue is reported because the 13-week experiment stops after 13 decisions; waiting bookings are not silently given extra call weeks.

In [4]:
selected_result = holdout_policy_results[SELECTED_POLICY]
selected_calls = selected_result["calls"]
selected_weekly = selected_result["weekly"]

display(
    selected_weekly[
        [
            "call_date",
            "eligible_queue",
            "calls",
            "cancellations_reached",
            "precision_at_capacity",
            "realized_loss_reached",
            "full_value_loss_reached",
            "average_days_to_arrival",
        ]
    ].style.format(
        {
            "precision_at_capacity": "{:.2%}",
            "realized_loss_reached": "{:,.0f}",
            "full_value_loss_reached": "{:,.0f}",
            "average_days_to_arrival": "{:.1f}",
        }
    )
)

,call_date,eligible_queue,calls,cancellations_reached,precision_at_capacity,realized_loss_reached,full_value_loss_reached,average_days_to_arrival
0,2016-12-05 00:00:00,611,150,105,70.00%,"7,089","53,464",122.8
1,2016-12-12 00:00:00,1270,150,114,76.00%,"4,795","71,472",121.9
2,2016-12-19 00:00:00,1753,150,99,66.00%,"7,856","37,867",100.1
3,2016-12-26 00:00:00,1910,150,72,48.00%,"7,797","33,995",96.6
4,2017-01-02 00:00:00,1969,150,67,44.67%,"8,280","44,589",104.2
5,2017-01-09 00:00:00,2653,150,85,56.67%,"12,898","70,976",110.7
6,2017-01-16 00:00:00,3268,150,90,60.00%,"20,902","76,321",100.4
7,2017-01-23 00:00:00,3964,150,87,58.00%,"13,313","62,454",116.4
8,2017-01-30 00:00:00,4560,150,86,57.33%,"8,595","67,298",113.6
9,2017-02-06 00:00:00,5032,150,89,59.33%,"11,040","67,017",111.1


## 3. Selected outreach policy and example call list

Each Monday, call the first 150 eligible rows after sorting by `p_cancel x max(adr, 0) x total_nights`. The index rises with estimated cancellation risk and with the amount of room revenue booked, so it favours bookings that are both likely to cancel and expensive to lose. A booking is eligible if it was made in an earlier week, has not been called before, has not already cancelled, and has not yet reached its arrival date. It leaves the standing queue once any of those stops being true.

The table below is an example from the first holdout Monday. `days_to_arrival_at_call` is shown for context and is not part of the ranking. `source_row_id` is only a reproducible source-row key, not a real reservation ID. The outcome column is deliberately absent because it would not be known when the list is produced.

In [5]:
first_call_date = selected_calls["call_date"].min()
example_call_list = selected_calls.loc[
    selected_calls["call_date"].eq(first_call_date),
    [
        "call_rank",
        "source_row_id",
        "hotel",
        "booking_date_proxy",
        "arrival_date",
        "p_cancel",
        "booking_value_proxy",
        "days_to_arrival_at_call",
        "risk_value_priority",
    ],
].head(10)

example_call_list.style.format(
    {
        "p_cancel": "{:.2%}",
        "booking_value_proxy": "{:,.0f}",
        "risk_value_priority": "{:,.1f}",
    }
)

,call_rank,source_row_id,hotel,booking_date_proxy,arrival_date,p_cancel,booking_value_proxy,days_to_arrival_at_call,risk_value_priority
0,1,64923,City Hotel,2016-11-29 00:00:00,2017-03-20 00:00:00,95.20%,"1,096",105,"1,043.0"
1,2,105221,City Hotel,2016-11-28 00:00:00,2017-01-24 00:00:00,76.09%,"1,309",50,996.1
2,3,62581,City Hotel,2016-11-28 00:00:00,2017-01-15 00:00:00,70.44%,"1,047",41,737.7
3,4,65804,City Hotel,2016-11-30 00:00:00,2017-04-08 00:00:00,79.69%,840,124,669.4
4,5,62330,City Hotel,2016-12-01 00:00:00,2017-01-08 00:00:00,64.01%,872,34,558.2
5,6,67771,City Hotel,2016-12-01 00:00:00,2017-05-06 00:00:00,68.68%,774,152,531.6
6,7,11683,Resort Hotel,2016-11-29 00:00:00,2017-05-25 00:00:00,83.58%,620,171,518.2
7,8,68284,City Hotel,2016-12-02 00:00:00,2017-05-15 00:00:00,99.86%,510,161,509.3
8,9,67869,City Hotel,2016-12-04 00:00:00,2017-05-08 00:00:00,72.42%,702,154,508.4
9,10,68193,City Hotel,2016-12-02 00:00:00,2017-05-15 00:00:00,99.84%,480,161,479.2


## 4. Illustrative quarterly return scenarios

The 13 holdout weeks are a natural next-quarter-sized evaluation. Historical outcomes tell us how much cancellation exposure the call list reached. They do not reveal whether a call and a $50 property credit would have changed any of those outcomes.

Two things vary in the grid, and it matters which is which.

**The save rate is an assumption about the program.** 5%, 10%, or 20% of the selected cancellations are prevented by being called. Nothing in this dataset supports any of these values; only a randomized pilot can.

**The loss basis is an assumption about the hotel.** It sets what a prevented cancellation was worth:

- *Linear resale recovery within 30 days* assumes a room canceled a month or more before arrival is resold at full price and cost nothing. This is the optimistic bound on recovery, and so the pessimistic bound on the program's value.
- *No resale at all* assumes every canceled room is a total loss. This is the pessimistic bound on recovery, and the optimistic bound on the program's value.

These two assumptions do different kinds of work, and it is worth being precise about which one decides the launch. Under the optimistic-recovery bound, no save rate in the grid reaches break-even: even preventing 20% of the cancellations called still loses money. Under the no-resale bound, break-even arrives at a save rate that is at least arguable. So the loss basis decides whether the program can work at all, while the save rate only decides how well it works once it can. Reality lies between the bounds and depends on occupancy, season, and channel mix.

Remaining assumptions are held fixed: contribution margin is 70%, each attempted call costs $5, and the average effective credit cost is either $25 or the full $50 per attempted call. The $25 case can represent only part of the offered credit becoming an economic cost because of contact, acceptance, redemption, or marginal-cost effects. The source file does not state the ADR currency; for this illustration ADR is assumed already converted into the same dollar-equivalent units as the call and credit costs, and the 2015-2017 values are not inflation-adjusted. The formula also assumes the same save probability for cancellations and no-shows. These are placeholders, not facts learned from the data.

In [6]:
loss_bases = {
    "realized_loss_proxy": "linear resale recovery within 30 days",
    "full_value_loss_proxy": "no resale at all",
}
scenario_rows = []

for loss_column, basis_label in loss_bases.items():
    for save_rate in [0.05, 0.10, 0.20]:
        for credit_cost in [25.0, 50.0]:
            estimate = evaluation.estimate_program_return(
                selected_calls,
                save_rate=save_rate,
                contribution_margin=0.70,
                call_cost=5.0,
                effective_credit_cost_per_call=credit_cost,
                loss_column=loss_column,
            )
            scenario_rows.append(
                {
                    "loss basis": basis_label,
                    "assumed save rate": save_rate,
                    "effective credit cost per call": credit_cost,
                    "estimated benefit": estimate["scenario_benefit"],
                    "estimated cost": estimate["scenario_cost"],
                    "estimated net return": estimate["net_return"],
                    "estimated ROI": estimate["roi"],
                    "break-even save rate": estimate[
                        "break_even_save_rate"
                    ],
                }
            )

return_scenarios = pd.DataFrame(scenario_rows)


def format_currency(value):
    sign = "-" if value < 0 else ""
    return f"{sign}${abs(value):,.0f}"


display(
    return_scenarios.set_index(
        ["loss basis", "assumed save rate", "effective credit cost per call"]
    ).style.format(
        {
            "estimated benefit": "${:,.0f}",
            "estimated cost": "${:,.0f}",
            "estimated net return": format_currency,
            "estimated ROI": "{:.1%}",
            "break-even save rate": "{:.1%}",
        }
    )
)

notice = selected_calls.loc[
    selected_calls["is_canceled"].eq(1), "cancellation_notice_days"
]
display(
    pd.Series(
        {
            "reached cancellations": int(notice.size),
            "median days of notice": notice.median(),
            "share cancelling within 30 days of arrival": (
                notice.lt(30).mean()
            ),
        },
        name="cancellation timing among calls made",
    ).to_frame()
)

,cancellation timing among calls made
reached cancellations,1146.000000
median days of notice,63.000000
share cancelling within 30 days of arrival,0.268761


## 5. Limitations and operational validation

- **No causal program effect:** there are no historical calls, offers, confirmations, or untreated controls. The save rate must be measured in a randomized pilot.
- **No measured resale rate:** the loss bounds are assumptions, not observations. This file records that a booking canceled and when, never what happened to the room afterwards. Because the two bounds straddle break-even, this single missing fact decides the launch question, and it is a question for hotel operations rather than for a model.
- **No true profit:** `ADR x nights` omits margin, documented currency, inflation, resale revenue, discounting, ancillary revenue, and the marginal cost of the $50 credit.
- **No contactability:** the file has no phone numbers, contact attempts, or response records, so the share of the 150 weekly calls that actually reach a guest is unknown.
- **Generalization:** these are two hotels from 2015-2017. Risk patterns, margins, and guest behavior may differ across properties, markets, and time periods.
- **Approximate time and retrospective status:** booking time is reconstructed from arrival date and lead time, has no time of day, and the file is not a true sequence of operational snapshots. Same-day ordering is ambiguous.
- **Limited scope:** conclusions cover complete booking cohorts with 0-180 days of lead time. They do not automatically extend to longer-lead bookings.
- **Queue simplifications:** each evaluation period starts without an incumbent backlog, model scores never update after booking, and some bookings remain queued after the thirteenth decision.
- **Untested call timing:** bookings become eligible on the Monday after their booking week. Eligibility does not guarantee a call: ranking and capacity can delay contact or leave a booking uncalled. Whether deliberately postponing calls until closer to arrival would work better is not answered here; it would require cancellation timing in the ranking objective and an explicit rule for holding bookings in the queue.
- **No real booking identifier:** `source_row_id` identifies a source row only. Exact matching rows are retained because separate anonymized bookings may look identical.

In [7]:
selected_label = policy_labels[SELECTED_POLICY]
reference_label = policy_labels[REFERENCE_POLICY]
selected_summary = holdout_summary.loc[selected_label]
risk_summary = holdout_summary.loc[reference_label]
random_mean = random_results[random_metric].mean()
selected_status_counts = selected_calls.loc[
    selected_calls["is_canceled"].eq(1), "reservation_status"
].value_counts()

resale_label = loss_bases["realized_loss_proxy"]
no_resale_label = loss_bases["full_value_loss_proxy"]


def scenario_value(basis, column, save_rate=None, credit_cost=25.0):
    rows = return_scenarios.loc[
        return_scenarios["loss basis"].eq(basis)
        & return_scenarios["effective credit cost per call"].eq(credit_cost)
    ]
    if save_rate is not None:
        rows = rows.loc[rows["assumed save rate"].eq(save_rate)]
    return rows[column].iloc[0]


break_even_resale = scenario_value(resale_label, "break-even save rate")
break_even_no_resale = scenario_value(no_resale_label, "break-even save rate")
break_even_no_resale_full_credit = scenario_value(
    no_resale_label, "break-even save rate", credit_cost=50.0
)
net_resale_10 = scenario_value(resale_label, "estimated net return", 0.10)
net_no_resale_10 = scenario_value(no_resale_label, "estimated net return", 0.10)
net_resale_20 = scenario_value(resale_label, "estimated net return", 0.20)
net_no_resale_20 = scenario_value(no_resale_label, "estimated net return", 0.20)
quarter_cost = scenario_value(no_resale_label, "estimated cost", 0.10)

reached_notice = selected_calls.loc[
    selected_calls["is_canceled"].eq(1), "cancellation_notice_days"
]
attrition = evaluation.pre_call_attrition(holdout_scored)
selected_aggregate = holdout_policy_results[SELECTED_POLICY]["aggregate"]

calls_made = int(selected_summary["calls"])
reached = int(selected_summary["cancellations reached"])
reference_reached = int(risk_summary["cancellations reached"])
never_called = int(selected_summary["bookings waiting after final decision"])
never_called_cancellations = int(
    selected_aggregate["eventual cancellations in final queue"]
)
canceled_before_first_call = int(attrition["canceled before first Monday"])

display(
    Markdown(
        f"""
## 6. Findings and recommendations

### Selected outreach policy

**Rank eligible bookings each Monday by cancellation risk times booked room value**
(`p_cancel x max(adr, 0) x total_nights`) and select up to 150. Eligible bookings were made in
an earlier week, have not been called, have not already canceled, and have not yet arrived.
Called bookings never return to the queue.

Over 13 holdout weeks, the selected policy fills **{calls_made:,} call slots** and reaches
**{reached:,} bookings that later canceled or became no-shows**
({int(selected_status_counts.get('Canceled', 0)):,} cancellations and
{int(selected_status_counts.get('No-Show', 0)):,} no-shows). Mean weekly precision@150 is
**{selected_summary['mean weekly precision@150']:.1%}**, against **{random_mean:.1%}** for random selection.
These are historical targeting results, not observed effects of outreach.

Risk-only ranking reaches more cancellation outcomes ({reference_reached:,} versus {reached:,}),
but value weighting reaches more canceled booked room value:
**{selected_summary['full value loss reached']:,.0f} ADR-equivalent units** versus
**{risk_summary['full value loss reached']:,.0f}**. The urgency-boosted variant was rejected on
validation because its small gain did not justify the added complexity (notebook 05).

### Financial sensitivity

**The scenarios span losses and gains because resale recovery and intervention effectiveness
remain unmeasured.** At {calls_made:,} attempts per quarter, assumed costs total
**{format_currency(quarter_cost)}** at a $25 effective credit cost and $5 staff cost per attempt.

| Room-resale assumption | Net return at 10% save rate | Net return at 20% save rate | Break-even save rate |
|---|---|---|---|
| Linear recovery within 30 days | {format_currency(net_resale_10)} | {format_currency(net_resale_20)} | **{break_even_resale:.1%}** |
| No resale | {format_currency(net_no_resale_10)} | {format_currency(net_no_resale_20)} | **{break_even_no_resale:.1%}** |

The calculation assumes 70% contribution margin and ADR converted into the same dollar-equivalent
units as program costs, without inflation adjustment. Save rates of 5%, 10%, and 20% are
illustrative; the source contains no evidence for any of them. Equal effectiveness for preventing
cancellations and no-shows is also an assumption.

Median cancellation notice among selected outcomes is **{reached_notice.median():.0f} days**,
and **{reached_notice.lt(30).mean():.1%}** occur within 30 days of arrival. Resale recovery therefore
changes how much of the booked exposure an intervention could protect. At a full $50 effective
credit cost, even the no-resale assumption requires **{break_even_no_resale_full_credit:.1%}**
effectiveness to break even.

### Validation needed for operational use

There are no historical calls, offers, contact records, or untreated controls. The data also
contains no actual room-resale recovery. Patterns from two hotels in 2015-2017 may differ across
properties, markets, and time periods; section 5 documents the remaining limitations.

Before extending this case study to a live program:

1. **Measure resale recovery by notice period** using occupancy and rate history.
2. **Collect genuine booking-time snapshots, reservation IDs, and contact records** to validate
   feature availability and replace source-row keys.
3. **Run a randomized pilot** among comparably ranked bookings, measuring stay completion,
   credit redemption and cost, handling time, and contribution margin.
4. **Measure contact and acceptance rates**, including credits issued to guests who would have
   stayed without outreach.

### Recommendation

Use the value-weighted ranking as a candidate for operational validation, measure resale economics,
and test outreach before committing to a broad rollout or a positive-return forecast.

The weekly cadence misses **{canceled_before_first_call:,}** holdout bookings that canceled before
their first eligible Monday. The quarter ends with **{never_called:,}** active bookings never called,
of which **{never_called_cancellations:,}** eventually cancel. Alternative contact schedules and
staffing levels require separate evaluation.
"""
    )
)



## 6. Findings and recommendations

### Selected outreach policy

**Rank eligible bookings each Monday by cancellation risk times booked room value**
(`p_cancel x max(adr, 0) x total_nights`) and select up to 150. Eligible bookings were made in
an earlier week, have not been called, have not already canceled, and have not yet arrived.
Called bookings never return to the queue.

Over 13 holdout weeks, the selected policy fills **1,950 call slots** and reaches
**1,146 bookings that later canceled or became no-shows**
(1,118 cancellations and
28 no-shows). Mean weekly precision@150 is
**58.8%**, against **26.0%** for random selection.
These are historical targeting results, not observed effects of outreach.

Risk-only ranking reaches more cancellation outcomes (1,278 versus 1,146),
but value weighting reaches more canceled booked room value:
**803,697 ADR-equivalent units** versus
**693,506**. The urgency-boosted variant was rejected on
validation because its small gain did not justify the added complexity (notebook 05).

### Financial sensitivity

**The scenarios span losses and gains because resale recovery and intervention effectiveness
remain unmeasured.** At 1,950 attempts per quarter, assumed costs total
**$58,500** at a $25 effective credit cost and $5 staff cost per attempt.

| Room-resale assumption | Net return at 10% save rate | Net return at 20% save rate | Break-even save rate |
|---|---|---|---|
| Linear recovery within 30 days | -$49,375 | -$40,251 | **64.1%** |
| No resale | -$2,241 | $54,018 | **10.4%** |

The calculation assumes 70% contribution margin and ADR converted into the same dollar-equivalent
units as program costs, without inflation adjustment. Save rates of 5%, 10%, and 20% are
illustrative; the source contains no evidence for any of them. Equal effectiveness for preventing
cancellations and no-shows is also an assumption.

Median cancellation notice among selected outcomes is **63 days**,
and **26.9%** occur within 30 days of arrival. Resale recovery therefore
changes how much of the booked exposure an intervention could protect. At a full $50 effective
credit cost, even the no-resale assumption requires **19.1%**
effectiveness to break even.

### Validation needed for operational use

There are no historical calls, offers, contact records, or untreated controls. The data also
contains no actual room-resale recovery. Patterns from two hotels in 2015-2017 may differ across
properties, markets, and time periods; section 5 documents the remaining limitations.

Before extending this case study to a live program:

1. **Measure resale recovery by notice period** using occupancy and rate history.
2. **Collect genuine booking-time snapshots, reservation IDs, and contact records** to validate
   feature availability and replace source-row keys.
3. **Run a randomized pilot** among comparably ranked bookings, measuring stay completion,
   credit redemption and cost, handling time, and contribution margin.
4. **Measure contact and acceptance rates**, including credits issued to guests who would have
   stayed without outreach.

### Recommendation

Use the value-weighted ranking as a candidate for operational validation, measure resale economics,
and test outreach before committing to a broad rollout or a positive-return forecast.

The weekly cadence misses **3,207** holdout bookings that canceled before
their first eligible Monday. The quarter ends with **5,621** active bookings never called,
of which **717** eventually cancel. Alternative contact schedules and
staffing levels require separate evaluation.
